# Task 3: Multinomial Naive Bayes (notebook version)

Notebook port of `task3_multinomial_nb.py`. The expensive `pd.read_csv` calls are isolated in their own cell and guarded by an `in globals()` check, so re-running downstream cells (cleaning, vectorizer, model, submission) does **not** re-read `train.csv` / `test.csv`.

**Typical workflow**
1. Run *Imports*, *Helpers*, *Config* once.
2. Run *Load data* once per kernel (force a reload by setting `FORCE_RELOAD = True` or deleting `train_df` / `test_df`).
3. Tweak `CONFIG` and re-run *Clean text*, *Validate*, *Full fit*, *Log run* as often as you like.

## Imports

In [1]:
import json
import os
import re
from datetime import datetime

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB

## Config

All parameters are in this code cell below, categorized. Text cleaning uses regex to preprocess the input and replace latex or URLs. Found to be useful and mostly left on.
Validation split controls the how much of the training set is used for testing, and contains the rng seed.
TFI DF Vectorizer controls all of its respective parameters, such as the number of features (words), min_df controls how many times a word needs to appear to be counted as a feature, whereas max_df controls what percentage of rows the feature must appear before it no longer is counted as a feature (for being non-distinguishing and not very useful due to commonality).
NGRAM Range is controls how many adjacent terms/features/words can be grouped as a single feature. These sequences are order dependent. Currently using Bigrams. Occassional arbitrary/stochastic testing of using trigrams yielded negative results, including trying with higher feature counts.

Sublinear TF. Apply log scaling to counts. Found stochastically after finding a local optimal without using it to bring improvements.

Alpha: Laplace scaling as explored in the homework. Originally only tested 0.1, 0.3, 0.5, 1.0. But after seeing some good results on some combos with 0.1 decided I should decrease the lower bound and explore lower values. Sweeps of these found 0.001 to be good.



In [75]:
CONFIG = {
    # I/O
    'DATA_DIR': './',
    'OUTPUT_FILE': 'MultinomialNB_Prediction.csv',
    'PARAMETERS_FILE': 'MultinomialNB_Parameters.json',
    'LOG_FILE': 'MultinomialNB_RunLog.txt',

    # Pipeline switches
    'CHECK_TRAINING_SCORE': False,
    'EXPORT_SUBMISSION': True,
    'STAMP_PREDICTION_FILE': True,
    'LOG_RUN': True,
    'SKIP_IF_ALREADY_LOGGED': True,

    # Text cleaning
    'USE_SAFE_TEXT_CLEANING': True,
    'STRIP_LATEX_MATH': True,
    'STRIP_URLS': True,
    'NORMALIZE_WHITESPACE': True,

    # Validation split
    'VAL_FRACTION': 0.2,
    'RANDOM_STATE': 42,

    # TfidfVectorizer
    'MAX_FEATURES': 750000,
    'MIN_DF': 2,
    'MAX_DF': 0.85,
    'STOP_WORDS': 'english',
    'NGRAM_RANGE': (1, 2),
    'SUBLINEAR_TF': False,

    # Naive Bayes model selection
    # 'multinomial' = MultinomialNB (uses ALPHA, FIT_PRIOR; ignores NORM)
    # 'complement'  = ComplementNB  (uses ALPHA, FIT_PRIOR, NORM)
    'MODEL': 'multinomial',
    'NORM': False,
    'ALPHA': 0.003,
    'FIT_PRIOR': False,
}

CONFIG

{'DATA_DIR': './',
 'OUTPUT_FILE': 'MultinomialNB_Prediction.csv',
 'PARAMETERS_FILE': 'MultinomialNB_Parameters.json',
 'LOG_FILE': 'MultinomialNB_RunLog.txt',
 'CHECK_TRAINING_SCORE': False,
 'EXPORT_SUBMISSION': True,
 'STAMP_PREDICTION_FILE': True,
 'LOG_RUN': True,
 'SKIP_IF_ALREADY_LOGGED': True,
 'USE_SAFE_TEXT_CLEANING': True,
 'STRIP_LATEX_MATH': True,
 'STRIP_URLS': True,
 'NORMALIZE_WHITESPACE': True,
 'VAL_FRACTION': 0.2,
 'RANDOM_STATE': 42,
 'MAX_FEATURES': 750000,
 'MIN_DF': 2,
 'MAX_DF': 0.85,
 'STOP_WORDS': 'english',
 'NGRAM_RANGE': (1, 2),
 'SUBLINEAR_TF': False,
 'MODEL': 'multinomial',
 'NORM': False,
 'ALPHA': 0.003,
 'FIT_PRIOR': False}

## Helpers

Pure functions: text cleaner, run-id, submission path builder, and the JSONL run-log dedup helpers. They all read from the `CONFIG` dict above so behaviour stays in sync with the config cell.

In [76]:
_LATEX_MATH_RE = re.compile(r'\$.*?\$')
_URL_RE = re.compile(r'http\S+')
_WHITESPACE_RE = re.compile(r'\s+')


def safe_clean_arxiv_text(text):
    if CONFIG['STRIP_LATEX_MATH']:
        text = _LATEX_MATH_RE.sub('LATEX_MATH_HERE', text)
    if CONFIG['STRIP_URLS']:
        text = _URL_RE.sub('URL_HERE', text)
    if CONFIG['NORMALIZE_WHITESPACE']:
        text = _WHITESPACE_RE.sub(' ', text).strip()
    return text


def get_editable_parameters():
    """Return the same JSON-serialisable parameter dict as the .py script."""
    params = dict(CONFIG)
    params['NGRAM_RANGE'] = list(CONFIG['NGRAM_RANGE'])
    params['RUN_PIPELINE'] = True
    params['EXPORT_PARAMETERS_ONLY'] = False
    return params


_DEDUP_KEYS = (
    'USE_SAFE_TEXT_CLEANING',
    'STRIP_LATEX_MATH',
    'STRIP_URLS',
    'NORMALIZE_WHITESPACE',
    'VAL_FRACTION',
    'RANDOM_STATE',
    'MAX_FEATURES',
    'MIN_DF',
    'MAX_DF',
    'STOP_WORDS',
    'NGRAM_RANGE',
    'SUBLINEAR_TF',
    'MODEL',
    'NORM',
    'ALPHA',
    'FIT_PRIOR',
)


def make_classifier():
    """Instantiate the NB estimator chosen by CONFIG['MODEL'].

    Centralised so the validation cell and the full-fit cell stay in sync, and
    so adding new model variants in future is one edit instead of two.
    """
    model = CONFIG['MODEL']
    if model == 'multinomial':
        return MultinomialNB(alpha=CONFIG['ALPHA'], fit_prior=CONFIG['FIT_PRIOR'])
    if model == 'complement':
        return ComplementNB(
            alpha=CONFIG['ALPHA'],
            fit_prior=CONFIG['FIT_PRIOR'],
            norm=CONFIG['NORM'],
        )
    raise ValueError(f"Unknown MODEL: {model!r}. Use 'multinomial' or 'complement'.")


def _params_signature(params):
    return tuple(str(params.get(key)) for key in _DEDUP_KEYS)


def find_logged_run(params=None):
    if params is None:
        params = get_editable_parameters()
    log_file = CONFIG['LOG_FILE']
    if not os.path.exists(log_file):
        return None
    target = _params_signature(params)
    match = None
    with open(log_file, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            if _params_signature(entry.get('parameters', {})) == target:
                match = entry
    return match


def make_run_id():
    return datetime.now().strftime('%Y%m%dT%H%M%S')


def build_submission_path(run_id):
    if not CONFIG['STAMP_PREDICTION_FILE']:
        return CONFIG['OUTPUT_FILE']
    base, ext = os.path.splitext(CONFIG['OUTPUT_FILE'])
    return f'{base}_{run_id}{ext}'


def append_run_log(run_id, val_macro_f1, train_accuracy, prediction_file):
    payload = {
        'run_id': run_id,
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'val_macro_f1': val_macro_f1,
        'train_accuracy': train_accuracy,
        'prediction_file': prediction_file,
        'parameters': get_editable_parameters(),
    }
    with open(CONFIG['LOG_FILE'], 'a', encoding='utf-8') as file:
        file.write(json.dumps(payload) + '\n')
    print(f"Appended run summary to {CONFIG['LOG_FILE']} (run_id={run_id})")

## Load data (run once per kernel)

The `if 'train_df' not in globals() or FORCE_RELOAD` guard makes this cell a no-op on subsequent re-executions, so you can safely *Run All* without paying for a full CSV reparse. To force a fresh load, either set `FORCE_RELOAD = True` here or `del train_df, test_df` from another cell.

In [77]:
FORCE_RELOAD = False

if FORCE_RELOAD or 'train_df' not in globals() or 'test_df' not in globals():
    print('Reading train.csv and test.csv ...')
    train_df = pd.read_csv(f"{CONFIG['DATA_DIR']}train.csv", sep='\t')
    test_df = pd.read_csv(f"{CONFIG['DATA_DIR']}test.csv", sep='\t')
    print(f'  train_df: {train_df.shape}')
    print(f'  test_df : {test_df.shape}')
else:
    print('Reusing cached train_df / test_df already in memory.')

X_text_raw = train_df['abstract'].fillna('').astype(str)
y = train_df['label_id'].astype(int)
X_test_text_raw = test_df['abstract'].fillna('').astype(str)

X_text_raw.head(2)

Reusing cached train_df / test_df already in memory.


0    Project-based learning plays a crucial role in...
1      Edge computing is a distributed computing pa...
Name: abstract, dtype: object

## Clean text

Re-run after toggling any of `USE_SAFE_TEXT_CLEANING`, `STRIP_LATEX_MATH`, `STRIP_URLS`, `NORMALIZE_WHITESPACE`. Output binds `X_text` / `X_test_text`, which the modeling cells consume.

In [78]:
cleaning_active = CONFIG['USE_SAFE_TEXT_CLEANING'] and (
    CONFIG['STRIP_LATEX_MATH']
    or CONFIG['STRIP_URLS']
    or CONFIG['NORMALIZE_WHITESPACE']
)

if cleaning_active:
    print('Applying safe_clean_arxiv_text ...')
    X_text = X_text_raw.apply(safe_clean_arxiv_text)
    X_test_text = X_test_text_raw.apply(safe_clean_arxiv_text)
else:
    print('Cleaning disabled; using raw abstracts.')
    X_text = X_text_raw
    X_test_text = X_test_text_raw

X_text.head(2)

Applying safe_clean_arxiv_text ...


0    Project-based learning plays a crucial role in...
1    Edge computing is a distributed computing para...
Name: abstract, dtype: object

## Dedup check (optional)

Mirrors `SKIP_IF_ALREADY_LOGGED` from the script. Just prints the matching prior run if any; it does **not** abort downstream cells (in a notebook you usually want to see numbers anyway).

In [79]:
if CONFIG['SKIP_IF_ALREADY_LOGGED']:
    prior = find_logged_run()
    if prior is not None:
        prior_f1 = prior.get('val_macro_f1')
        prior_acc = prior.get('train_accuracy')
        prior_run_id = prior.get('run_id') or prior.get('timestamp')
        prior_pred = prior.get('prediction_file') or 'none'
        f1_str = f'{prior_f1:.4f}' if isinstance(prior_f1, (int, float)) else 'n/a'
        acc_str = f'{prior_acc:.4f}' if isinstance(prior_acc, (int, float)) else 'n/a'
        print(
            'Matching parameter set already logged:\n'
            f'  prior run_id        : {prior_run_id}\n'
            f'  prior val_macro_f1  : {f1_str}\n'
            f'  prior train_accuracy: {acc_str}\n'
            f'  prior prediction    : {prior_pred}'
        )
    else:
        print('No prior log entry matches the current CONFIG.')
else:
    print('SKIP_IF_ALREADY_LOGGED is False; skipping dedup check.')

No prior log entry matches the current CONFIG.


## Validate (held-out split)

Skipped when `VAL_FRACTION == 0`. Builds a *separate* TF-IDF vocabulary from the train split only, so validation numbers don't leak the val rows into vectorizer fitting.

In [80]:
val_macro_f1 = None

if CONFIG['VAL_FRACTION'] > 0:
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_text,
        y,
        test_size=CONFIG['VAL_FRACTION'],
        random_state=CONFIG['RANDOM_STATE'],
        stratify=y,
    )
    vec_val = TfidfVectorizer(
        max_features=CONFIG['MAX_FEATURES'],
        min_df=CONFIG['MIN_DF'],
        max_df=CONFIG['MAX_DF'],
        stop_words=CONFIG['STOP_WORDS'],
        ngram_range=CONFIG['NGRAM_RANGE'],
        sublinear_tf=CONFIG['SUBLINEAR_TF'],
    )
    X_tr_vec = vec_val.fit_transform(X_tr)

    clf_val = make_classifier()
    clf_val.fit(X_tr_vec, y_tr)

    y_hat = clf_val.predict(vec_val.transform(X_va))
    val_macro_f1 = float(f1_score(y_va, y_hat, average='macro'))
    print(f'Validation macro-F1: {val_macro_f1:.4f}')
else:
    print('VAL_FRACTION is 0; skipping validation.')

Validation macro-F1: 0.6549


## Full fit + (optional) submission

Refits TF-IDF + the chosen Naive Bayes model (`MODEL` in `CONFIG`, either `'multinomial'` or `'complement'`) on the entire training set, then optionally scores on train and writes a Kaggle CSV. Skipped entirely when both `CHECK_TRAINING_SCORE` and `EXPORT_SUBMISSION` are False.

In [81]:
run_id = make_run_id()
train_accuracy = None
prediction_file = None

needs_full_fit = CONFIG['EXPORT_SUBMISSION'] or CONFIG['CHECK_TRAINING_SCORE']

if needs_full_fit:
    vectorizer = TfidfVectorizer(
        max_features=CONFIG['MAX_FEATURES'],
        min_df=CONFIG['MIN_DF'],
        max_df=CONFIG['MAX_DF'],
        stop_words=CONFIG['STOP_WORDS'],
        ngram_range=CONFIG['NGRAM_RANGE'],
        sublinear_tf=CONFIG['SUBLINEAR_TF'],
    )
    X_full_vec = vectorizer.fit_transform(X_text)

    clf = make_classifier()
    clf.fit(X_full_vec, y)

    if CONFIG['CHECK_TRAINING_SCORE']:
        train_accuracy = float(clf.score(X_full_vec, y))
        print(f'Training accuracy: {train_accuracy:.4f}')

    if CONFIG['EXPORT_SUBMISSION']:
        X_test_vec = vectorizer.transform(X_test_text)
        test_preds = clf.predict(X_test_vec)

        submission = pd.DataFrame({
            'id': test_df['id'].astype(int),
            'label_id': test_preds.astype(int),
        })
        prediction_file = build_submission_path(run_id)
        submission.to_csv(prediction_file, index=False)
        print(f'Saved Kaggle submission to {prediction_file} (run_id={run_id})')
else:
    print('Skipped full retrain (EXPORT_SUBMISSION and CHECK_TRAINING_SCORE are False).')

Saved Kaggle submission to MultinomialNB_Prediction_20260420T161337.csv (run_id=20260420T161337)


## Log run

Appends one JSONL line to `MultinomialNB_RunLog.txt` containing the `run_id`, val score, train accuracy, prediction filename, and the full editable-parameter snapshot. Skip by setting `LOG_RUN` to False in `CONFIG`.

In [82]:
if CONFIG['LOG_RUN']:
    append_run_log(run_id, val_macro_f1, train_accuracy, prediction_file)
else:
    print('LOG_RUN is False; not writing to run log.')

Appended run summary to MultinomialNB_RunLog.txt (run_id=20260420T161337)


## Export parameters JSON (optional)

Equivalent of the `.py` script's `EXPORT_PARAMETERS_ONLY` short-circuit: write the current editable parameters to `MultinomialNB_Parameters.json`.

In [83]:
with open(CONFIG['PARAMETERS_FILE'], 'w', encoding='utf-8') as file:
    json.dump(get_editable_parameters(), file, indent=2)
print(f"Saved editable parameters to {CONFIG['PARAMETERS_FILE']}")

Saved editable parameters to MultinomialNB_Parameters.json
